In [ ]:
import requests
import pandas as pd

# Define search terms
neanderthal_terms = ['neanderthal', 'neanderthals', 'neandertal', 'neandertals', '"middle paleolithic"']
site_terms = ['site', 'sites', 'locality', 'localities', 'assemblage', 'assemblages']

# Construct the Europe PMC query
# We'll build a query that searches for papers where both a Neanderthal term and a site term are present in the title or abstract
neanderthal_query = ' OR '.join([f'TITLE:{term} OR ABSTRACT:{term}' for term in neanderthal_terms])
site_query = ' OR '.join([f'TITLE:{term} OR ABSTRACT:{term}' for term in site_terms])

combined_query = f'({neanderthal_query}) AND ({site_query})'

# The Europe PMC API endpoint
base_url = 'https://www.ebi.ac.uk/europepmc/webservices/rest/search'

# Parameters for the API request
params = {
    'query': combined_query,
    'format': 'json',
    'pageSize': 1000,  # Max page size
    'cursorMark': '*',
    'resultType': 'core',
}

all_results = []
while True:
    response = requests.get(base_url, params=params)
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        break

    data = response.json()
    results = data.get('resultList', {}).get('result', [])
    all_results.extend(results)

    # Check if there are more results
    next_cursor_mark = data.get('nextCursorMark')
    if not next_cursor_mark or next_cursor_mark == params['cursorMark']:
        break

    params['cursorMark'] = next_cursor_mark

print(f"Total records retrieved: {len(all_results)}")

# Create DataFrame
df = pd.DataFrame(all_results)



In [3]:
# Keep relevant columns
df = df[['id', 'title', 'authorString', 'pubYear', 'doi', 'abstractText']]

# Remove duplicates based on DOI
df = df.drop_duplicates(subset='doi')

# Function to check if text contains terms
def contains_terms(text, terms):
    if pd.isna(text):
        return False
    text_lower = text.lower()
    return any(term.strip('"').lower() in text_lower for term in terms)

# Filter DataFrame
mask = (
    (df['title'].apply(lambda x: contains_terms(x, neanderthal_terms) and contains_terms(x, site_terms))) |
    (df['abstractText'].apply(lambda x: contains_terms(x, neanderthal_terms) and contains_terms(x, site_terms))) |
    (df['title'].apply(lambda x: contains_terms(x, neanderthal_terms)) & df['abstractText'].apply(lambda x: contains_terms(x, site_terms))) |
    (df['title'].apply(lambda x: contains_terms(x, site_terms)) & df['abstractText'].apply(lambda x: contains_terms(x, neanderthal_terms)))
)

filtered_df = df[mask].reset_index(drop=True)

print(f"Filtered records: {len(filtered_df)}")

# Display the first few records
print(filtered_df.head())

# Optionally, save the DataFrame to a CSV file
filtered_df.to_csv('europe_pmc_neanderthal_sites.csv', index=False)


Filtered records: 617
          id                                              title  \
0   39265525  Long genetic and social isolation in Neanderth...   
1   37315040  On the Quina side: A Neanderthal bone industry...   
2  PPR660653  On the Quina side: A Neanderthal bone industry...   
3   39271648  Chronometric data and stratigraphic evidence s...   
4   38565571  The Neanderthal niche space of Western Eurasia...   

                                        authorString pubYear  \
0  Slimak L, Vimala T, Seguin-Orlando A, Metz L, ...    2024   
1  Baumann M, Plisson H, Maury S, Renou S, Coqueu...    2023   
2  Baumann M, Plisson H, Maury S, Renou S, Coqueu...    2023   
3  Higham T, Frouin M, Douka K, Ronchitelli A, Bo...    2024   
4               Yaworsky PM, Nielsen ES, Nielsen TK.    2024   

                            doi  \
0    10.1016/j.xgen.2024.100593   
1  10.1371/journal.pone.0284081   
2     10.1101/2023.05.15.540772   
3    10.1038/s41467-024-51546-9   
4    10.1038/s4

In [4]:
filtered_df

,id,title,authorString,pubYear,doi,abstractText
0,39265525,Long genetic and social isolation in Neanderth...,"Slimak L, Vimala T, Seguin-Orlando A, Metz L, ...",2024,10.1016/j.xgen.2024.100593,Neanderthal genomes have been recovered from s...
1,37315040,On the Quina side: A Neanderthal bone industry...,"Baumann M, Plisson H, Maury S, Renou S, Coqueu...",2023,10.1371/journal.pone.0284081,Did Neanderthal produce a bone industry? The r...
2,PPR660653,On the Quina side: A Neanderthal bone industry...,"Baumann M, Plisson H, Maury S, Renou S, Coqueu...",2023,10.1101/2023.05.15.540772,Did Neanderthal produce a bone industry? The r...
3,39271648,Chronometric data and stratigraphic evidence s...,"Higham T, Frouin M, Douka K, Ronchitelli A, Bo...",2024,10.1038/s41467-024-51546-9,The process by which Palaeolithic Europe was t...
4,38565571,The Neanderthal niche space of Western Eurasia...,"Yaworsky PM, Nielsen ES, Nielsen TK.",2024,10.1038/s41598-024-57490-4,Neanderthals occupied Western Eurasia between ...
...,...,...,...,...,...,...
612,28557099,Age assessment of the Spitalfields cemetery po...,Loth SR.,1995,10.1002/ajhb.1310070408,Accurate paleodemographic reconstruction depen...
613,7785727,Enamel hypoplasia in the middle pleistocene ho...,"Bermúdez de Castro JM, Pérez PJ.",1995,10.1002/ajpa.1330960307,The prevalence and chronology of enamel hypopl...
614,813529,A fossil hominid frontal from Velika Pećina (C...,Smith FH.,1976,10.1002/ajpa.1330440118,Fossil hominid remains dating to the Upper Ple...
615,16063324,A new Neandertal child mandible from an Upper ...,"Ascenzi A, Segre AG.",1971,10.1038/233280a0,NaN


In [ ]:
filtered_df